[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Missing Data &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Find the first and last hour of Svalbard's gap from the index of its missing readings.

In [2]:
svalbard = readings[readings["station"] == "Svalbard"].set_index("time")["celsius"]
gap = svalbard[svalbard.isna()].index

print(gap.min(), "to", gap.max())

2025-03-02 00:00:00 to 2025-03-02 23:00:00


The mask keeps the 24 missing hours, and their index holds the times, from midnight to 23:00 on 2
March.

**2.** Print Svalbard's mean for March with the gaps left out, and with `skipna=False`.

In [3]:
svalbard = readings[readings["station"] == "Svalbard"].set_index("time")["celsius"]
march = svalbard.loc["2025-03"]

print(march.count(), "readings |", round(march.mean(), 2), march.mean(skipna=False))

720 readings | -9.28 nan


With the gaps left out, the mean is of March's 720 real readings. With `skipna=False`, one missing
value makes the mean missing.

**3.** Fill Svalbard's gap with `interpolate`, and print the filled readings at 06:00, 12:00 and
18:00 on 2 March.

In [4]:
svalbard = readings[readings["station"] == "Svalbard"].set_index("time")["celsius"]
filled = svalbard.interpolate()

filled.loc[["2025-03-02 06:00", "2025-03-02 12:00", "2025-03-02 18:00"]]

time
2025-03-02 06:00:00   -12.42
2025-03-02 12:00:00   -12.78
2025-03-02 18:00:00   -13.14
Name: celsius, dtype: float64

Each is a point on the straight line from -12.0 the night before to -13.5 the midnight after, so the
values fall steadily through the day.

**4.** Rebuild the small logbook, and keep only the rows that have a wind speed, using `dropna` with
`subset`.

In [5]:
log = pd.DataFrame({
    "station": ["Oslo", "Oslo", "Bergen", None],
    "celsius": [1.2, None, 2.5, None],
    "wind": [None, None, 3.0, None],
})

log.dropna(subset=["wind"])

,station,celsius,wind
2,Bergen,2.5,3.0


Only Bergen's row has a wind speed, so it is the one row left.

**5.** Make a Series of the whole numbers 4, 6 and 9 with a gap after the 6, as `Int64`, and print
its sum and its mean.

In [6]:
numbers = pd.Series([4, 6, None, 9], dtype="Int64")

print(numbers)
print(numbers.sum(), round(numbers.mean(), 2))

0       4
1       6
2    <NA>
3       9
dtype: Int64
19 6.33


The sum and the mean skip the gap, so the mean is of three numbers, 19 divided by 3.

---

&#8592; **Back to:** [Missing Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/10-missing-data.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
